In [25]:
import torch
import torch.nn as nn 
from torchvision import models, transforms
import torchvision

In [ ]:
# 1. Inisialisasi Model & Ekstraksi Bobot Pretrained
weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights)

In [ ]:
# 2. Konfirmasi B2: Feature Extraction
for param in model.parameters():
    param.requires_grad = False

In [ ]:
# Ganti classifier head untuk 10 kelas
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

In [18]:
# 3. Konfigurasi B3: Unfreeze blok residual terakhir (layer4)
for param in model.layer4.parameters():
    param.requires_grad = True

In [19]:
# Optimasi dengan Differential Learning Rate
optimizer = torch.optim.AdamW([
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
], weight_decay=1e-2)

In [20]:
# 4. Pipeline Transformasi data
# Diterapkan augmentasi aktif pada data training
train_transforms = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=weights.transforms().mean, std=weights.transforms().std
        ),
    ]
)

In [22]:
# Data validasi, tidak didistorsi
eval_transforms = weights.transforms()

In [23]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable Parameters: {trainable_params:,} / {total_params:,}")

Trainable Parameters: 8,398,858 / 11,181,642


In [ ]:
# Hyperparameters
batch_size = 32
epochs = 200
patience = 5
seed = 42
data_augmentation = True
num_classes = 10
subtract_pixel_mean = True

In [ ]:
dataset = torchvision.datasets.OxfordIIITPet('./', download=True)
# input_shape = x_train.shape[1:]

Dataset OxfordIIITPet
    Number of datapoints: 3680
    Root location: ./

In [ ]:
x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255